In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import ast
import json

In [3]:
df = pd.read_csv("courses_dataset_final.csv")
df["prerequisites"] = df["prerequisites"].apply(ast.literal_eval)
print(df.shape)
df.head(10)

(2100, 14)


,course_id,course_name,category,skills,level,duration_hours,rating,instructor,description,enrolled_students,price,language,tags,prerequisites
0,1,"D3.js - Advanced Dashboards (Live Cohort, Data...",Data Visualization,d3.js advanced dashboards data visualization,Advanced,20,4.5,Prof. J. Mehta,Learn D3.js through a advanced dashboards focu...,18191,0,Hindi,"trending, project-based",[1503]
1,2,"PHP - Data Structures (Crash Course, LinkedIn ...",Programming,php data structures programming,Beginner,8,4.8,Prof. J. Mehta,Learn PHP through a data structures focused co...,23921,1999,English + Hindi Subtitles,"beginner-friendly, project-based, job-ready, q...",[]
2,3,Pandas - Practical Applications (Project-Based...,Data Science,pandas practical applications data science,Intermediate,10,3.7,Ms. K. Verma,Learn Pandas through a practical applications ...,9752,1999,Spanish,"industry-recognized, hands-on, in-demand",[1697]
3,4,"Ruby - Advanced Concepts (Crash Course, Plural...",Programming,ruby advanced concepts programming,Advanced,10,3.7,Dr. A. Sharma,Learn Ruby through a advanced concepts focused...,17151,2499,French,"trending, mentor-support, job-ready, project-b...",[568]
4,5,Statistical Analysis - Best Practices (Live Co...,Data Science,statistical analysis best practices data science,Beginner,12,4.5,Prof. J. Mehta,Learn Statistical Analysis through a best prac...,32989,2499,French,"mentor-support, job-ready, beginner-friendly",[]
5,6,"Java - Basics (Self-Paced, Coursera)",Programming,java basics programming,Beginner,25,4.2,Mr. T. Singh,"Learn Java through a basics focused course, co...",25045,0,English,"industry-recognized, project-based, in-demand",[]
6,7,Resume Building - Practical Guide (Crash Cours...,Career Skills,resume building practical guide career skills,Beginner,6,3.9,Prof. R. Iyer,Learn Resume Building through a practical guid...,42748,1499,English + Hindi Subtitles,"job-ready, hands-on, industry-recognized",[]
7,8,Feature Engineering - Optimization Strategies ...,Machine Learning,feature engineering optimization strategies ma...,Advanced,8,4.0,Dr. A. Sharma,Learn Feature Engineering through a optimizati...,26265,499,English + Hindi Subtitles,"certificate, project-based, in-demand, mentor-...",[1602]
8,9,Matplotlib - Real World Projects (Project-Base...,Data Visualization,matplotlib real world projects data visualization,Beginner,20,4.4,Mr. S. Gupta,Learn Matplotlib through a real world projects...,16186,1999,Hindi,"industry-recognized, in-demand, job-ready, cer...",[]
9,10,"SQL - Real World Projects (Self-Paced, edX)",Databases,sql real world projects databases,Beginner,30,3.9,Dr. N. Rao,Learn SQL through a real world projects focuse...,6137,1499,English,"trending, project-based",[]


In [4]:
vectorizer = TfidfVectorizer()
course_vectors = vectorizer.fit_transform(df["skills"])
print("TF-IDF matrix shape:", course_vectors.shape)

TF-IDF matrix shape: (2100, 206)


In [11]:
# Kmeans model training

In [12]:
NUM_CLUSTERS = 11
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(course_vectors)

print(df["cluster"].value_counts())

cluster
2     304
1     240
8     223
6     221
10    177
5     169
7     165
9     163
0     158
3     156
4     124
Name: count, dtype: int64


In [13]:
# Skill gap Analyzer

In [14]:
SKILL_LEVEL_MAP = {"None": 0, "Basic": 1, "Beginner": 1, "Medium": 2, "Intermediate": 2, "High": 3, "Advanced": 3}

def analyze_skill_gap(user_skills, required_skills):
    gap_report = {}
    for skill in required_skills:
        current_level = SKILL_LEVEL_MAP.get(user_skills.get(skill, "None"), 0)
        gap_score = 3 - current_level
        gap_report[skill] = {
            "current_level": user_skills.get(skill, "None"),
            "gap_score": gap_score,
            "priority": "High" if gap_score >= 2 else ("Medium" if gap_score == 1 else "Low")
        }
    return gap_report

In [15]:
def recommend_courses(user_interests, top_n=5):
    user_vector = vectorizer.transform([user_interests])
    predicted_cluster = kmeans.predict(user_vector)[0]

    cluster_df = df[df["cluster"] == predicted_cluster].copy()
    cluster_vectors = vectorizer.transform(cluster_df["skills"])
    similarity_scores = cosine_similarity(user_vector, cluster_vectors).flatten()
    cluster_df["similarity"] = similarity_scores

    ranked = cluster_df.sort_values(by=["similarity", "rating"], ascending=[False, False]).head(top_n)
    return ranked[["course_id", "course_name", "level", "rating", "similarity", "cluster"]].to_dict(orient="records")

In [16]:
def build_roadmap(course_ids):
    course_map = df.set_index("course_id").to_dict(orient="index")
    visited = []
    result = []
    def visit(cid):
        if cid in visited or cid not in course_map:
            return
        visited.append(cid)
        for prereq in course_map[cid]["prerequisites"]:
            visit(prereq)
        result.append(cid)
    for cid in course_ids:
        visit(cid)
    roadmap = []
    for i, cid in enumerate(result, start=1):
        roadmap.append({
            "step": i, "course_id": cid,
            "course_name": course_map[cid]["course_name"],
            "level": course_map[cid]["level"]
        })
    return roadmap

In [17]:
def check_unlock_next(current_score, threshold=75.0):
    unlocked = current_score >= threshold
    return {
        "score": current_score, "threshold": threshold,
        "unlocked_next_module": unlocked,
        "message": "Great job! Next module unlocked." if unlocked
                    else f"Score {current_score}% is below {threshold}%. Revise and retry."
    }

In [18]:
def generate_learning_path(user_profile):
    skill_gap = analyze_skill_gap(user_profile["current_skills"], user_profile["required_skills"])
    recommended = recommend_courses(user_profile["interests"], top_n=5)
    course_ids = [c["course_id"] for c in recommended]
    roadmap = build_roadmap(course_ids)
    return {
        "user": user_profile["name"],
        "goal": user_profile["goal"],
        "skill_gap_analysis": skill_gap,
        "recommended_courses": recommended,
        "learning_roadmap": roadmap
    }

In [19]:
sample_user = {
    "name": "Aarav Sharma",
    "goal": "Data Scientist",
    "current_skills": {"Python": "Basic", "SQL": "Beginner", "Statistics": "None"},
    "interests": "python data analysis machine learning statistics",
    "required_skills": ["Python", "Statistics", "SQL", "Machine Learning"]
}

output = generate_learning_path(sample_user)
print(json.dumps(output, indent=2))

{
  "user": "Aarav Sharma",
  "goal": "Data Scientist",
  "skill_gap_analysis": {
    "Python": {
      "current_level": "Basic",
      "gap_score": 2,
      "priority": "High"
    },
    "Statistics": {
      "current_level": "None",
      "gap_score": 3,
      "priority": "High"
    },
    "SQL": {
      "current_level": "Beginner",
      "gap_score": 2,
      "priority": "High"
    },
    "Machine Learning": {
      "current_level": "None",
      "gap_score": 3,
      "priority": "High"
    }
  },
  "recommended_courses": [
    {
      "course_id": 1488,
      "course_name": "Supervised Learning - Fundamentals (Crash Course, Udemy)",
      "level": "Beginner",
      "rating": 4.7,
      "similarity": 0.29622499635258825,
      "cluster": 8
    },
    {
      "course_id": 1978,
      "course_name": "Supervised Learning - Fundamentals (Self-Paced, Coursera)",
      "level": "Beginner",
      "rating": 4.3,
      "similarity": 0.29622499635258825,
      "cluster": 8
    },
    {
      